# Customer Intelligence & Churn Prevention Strategy## Olist E-Commerce Marketplace — 100K+ Transactions (2016–2018)---### Executive Summary**Business Problem:** Olist, Brazil's largest e-commerce marketplace, faces a critical retention challenge — over 97% of customers make only a single purchase, leaving significant lifetime value unrealised.**Objective:** Build an end-to-end customer intelligence pipeline that segments the customer base, predicts churn risk, and delivers actionable retention strategies with quantified revenue impact.**Key Findings:**- The top 20% of customers drive ~80% of total revenue (classic Pareto), but most are single-purchase buyers- K-Means clustering reveals 4 distinct customer segments with different retention needs- A Random Forest churn model achieves strong predictive performance, identifying at-risk customers before they lapse- Targeted retention of high-value at-risk customers could recover significant revenue at 3–5x ROI**Methods:** RFM analysis, K-Means clustering (with silhouette validation), cohort retention analysis, logistic regression & random forest churn prediction, CLV modelling, revenue impact quantification---*Author: Lily | MSc Student, UCL*  *Tools: Python, pandas, scikit-learn, matplotlib, seaborn*

## 1. Data Loading & PreparationWe work with 5 interconnected Olist datasets covering orders, customers, items, payments, and products. The goal of this section is to merge them into a single analytical dataset, handle missing values, and engineer features for downstream modelling.

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport seaborn as snsfrom datetime import timedeltaimport warningswarnings.filterwarnings("ignore")# Visual style — consistent throughout the notebooksns.set_style("whitegrid")plt.rcParams.update({    "figure.figsize": (12, 6),    "font.size": 11,    "axes.titlesize": 14,    "axes.labelsize": 12,    "axes.titleweight": "bold"})PALETTE = ["#2E86AB", "#A23B72", "#F18F01", "#C73E1D", "#3B1F2B"]print("Libraries loaded successfully")

In [ ]:
# ==============================# 1.1 Load all datasets# ==============================orders     = pd.read_csv("olist_orders_dataset.csv")customers  = pd.read_csv("olist_customers_dataset.csv")items      = pd.read_csv("olist_order_items_dataset.csv")payments   = pd.read_csv("olist_payments_dataset.csv")products   = pd.read_csv("olist_products_dataset.csv")datasets = {"Orders": orders, "Customers": customers, "Items": items,            "Payments": payments, "Products": products}print("Dataset overview:")print("-" * 45)for name, d in datasets.items():    print(f"  {name:12s} → {d.shape[0]:>7,} rows × {d.shape[1]:>2} cols")print(f"\nTotal records across all tables: {sum(d.shape[0] for d in datasets.values()):,}")

In [ ]:
# ==============================# 1.2 Data quality checks# ==============================# Datetime conversiondate_cols = ["order_purchase_timestamp", "order_approved_at",             "order_delivered_carrier_date", "order_delivered_customer_date",             "order_estimated_delivery_date"]for col in date_cols:    orders[col] = pd.to_datetime(orders[col])# Missing values summaryprint("Missing values (columns with any nulls):")print("-" * 45)for name, d in datasets.items():    nulls = d.isnull().sum()    nulls = nulls[nulls > 0]    if len(nulls) > 0:        for col, count in nulls.items():            print(f"  {name:12s} | {col:35s} | {count:>5} ({count/len(d)*100:.1f}%)")# Duplicatesprint("\nDuplicates removed:")for name, d in datasets.items():    n_dup = d.duplicated().sum()    if n_dup > 0:        print(f"  {name}: {n_dup}")    datasets[name] = d.drop_duplicates()orders, customers, items, payments, products = datasets.values()

In [ ]:
# ==============================# 1.3 Merge into master analytical dataset# ==============================# Only keep delivered orders (completed transactions)delivered = orders[orders["order_status"] == "delivered"].copy()print(f"Filtering to delivered orders: {len(delivered):,} / {len(orders):,} "      f"({len(delivered)/len(orders)*100:.1f}%)")df = (delivered      .merge(customers, on="customer_id", how="left")      .merge(items, on="order_id", how="left")      .merge(payments, on="order_id", how="left")      .merge(products[["product_id", "product_category_name"]], on="product_id", how="left"))# Feature engineeringdf["order_date"] = df["order_purchase_timestamp"].dt.datedf["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")df["order_dow"] = df["order_purchase_timestamp"].dt.day_name()df["order_hour"] = df["order_purchase_timestamp"].dt.hourdf["delivery_days"] = (df["order_delivered_customer_date"] - df["order_purchase_timestamp"]).dt.daysprint(f"\nMaster dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")print(f"Date range: {df['order_purchase_timestamp'].min().date()} → {df['order_purchase_timestamp'].max().date()}")print(f"Unique customers: {df['customer_unique_id'].nunique():,}")

## 2. Exploratory Data AnalysisBefore modelling, we need to understand the business landscape: revenue trends, customer behaviour patterns, and where the opportunities lie. Every chart below is annotated with the business implication — not just what the data shows, but *what it means*.

In [ ]:
# ==============================# 2.1 Revenue & order trends# ==============================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Monthly GMVgmv = df.groupby("order_month")["payment_value"].sum()axes[0].plot(gmv.index.astype(str), gmv.values, marker="o", color=PALETTE[0], linewidth=2)axes[0].fill_between(range(len(gmv)), gmv.values, alpha=0.1, color=PALETTE[0])axes[0].set_title("Monthly gross merchandise value (GMV)")axes[0].set_ylabel("Revenue (R$)")axes[0].tick_params(axis="x", rotation=45)# Annotate peakpeak_idx = gmv.values.argmax()axes[0].annotate(f"Peak: R${gmv.values[peak_idx]:,.0f}",                 xy=(peak_idx, gmv.values[peak_idx]),                 xytext=(peak_idx-3, gmv.values[peak_idx]*1.1),                 arrowprops=dict(arrowstyle="->", color=PALETTE[3]),                 fontsize=10, color=PALETTE[3], fontweight="bold")# Monthly ordersorders_m = df.groupby("order_month")["order_id"].nunique()axes[1].plot(orders_m.index.astype(str), orders_m.values, marker="o", color=PALETTE[1], linewidth=2)axes[1].fill_between(range(len(orders_m)), orders_m.values, alpha=0.1, color=PALETTE[1])axes[1].set_title("Monthly order volume")axes[1].set_ylabel("Number of orders")axes[1].tick_params(axis="x", rotation=45)plt.tight_layout()plt.savefig("fig_revenue_trends.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n💡 INSIGHT: Revenue grew {((gmv.iloc[-2] / gmv.iloc[1]) - 1)*100:.0f}% over the period,")print(f"   but the growth rate is flattening — suggesting organic acquisition alone won't sustain momentum.")

In [ ]:
# ==============================# 2.2 Customer spending distribution & top categories# ==============================fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Spending distribution (trimmed)user_spending = df.groupby("customer_unique_id")["payment_value"].sum()q99 = user_spending.quantile(0.99)filtered = user_spending[user_spending <= q99]axes[0].hist(filtered, bins=50, color=PALETTE[0], alpha=0.7, edgecolor="white")axes[0].axvline(user_spending.median(), color=PALETTE[3], linestyle="--", linewidth=2,                label=f"Median: R${user_spending.median():.0f}")axes[0].axvline(user_spending.mean(), color=PALETTE[2], linestyle="--", linewidth=2,                label=f"Mean: R${user_spending.mean():.0f}")axes[0].set_title("Customer spending distribution")axes[0].set_xlabel("Total spend (R$)")axes[0].set_ylabel("Number of customers")axes[0].legend()# Top 10 categoriestop_cats = (df.groupby("product_category_name")["payment_value"]              .sum().sort_values(ascending=True).tail(10))axes[1].barh(top_cats.index, top_cats.values, color=PALETTE[0], alpha=0.8)axes[1].set_title("Top 10 product categories by revenue")axes[1].set_xlabel("Revenue (R$)")plt.tight_layout()plt.savefig("fig_spending_distribution.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n💡 INSIGHT: Median spend is R${user_spending.median():.0f} vs mean R${user_spending.mean():.0f}")print(f"   → Heavy right skew — a small number of high-value customers pull the mean up.")print(f"   This is exactly where segmentation adds value.")

In [ ]:
# ==============================# 2.3 Repeat purchase behaviour — THE critical metric# ==============================customer_stats = df.groupby("customer_unique_id").agg(    n_orders=("order_id", "nunique"),    total_spend=("payment_value", "sum"),    first_purchase=("order_purchase_timestamp", "min"),    last_purchase=("order_purchase_timestamp", "max"))customer_stats["lifetime_days"] = (customer_stats["last_purchase"] - customer_stats["first_purchase"]).dt.daysrepeat_rate = (customer_stats["n_orders"] > 1).mean()fig, axes = plt.subplots(1, 2, figsize=(14, 5))# Order frequencyorder_dist = customer_stats["n_orders"].value_counts().sort_index().head(8)colors = [PALETTE[3] if x == 1 else PALETTE[0] for x in order_dist.index]axes[0].bar(order_dist.index.astype(str), order_dist.values, color=colors, alpha=0.8)axes[0].set_title("Orders per customer")axes[0].set_xlabel("Number of orders")axes[0].set_ylabel("Number of customers")# Add the critical stataxes[0].annotate(f"{(1-repeat_rate)*100:.1f}% buy\nonly ONCE",                 xy=(0, order_dist.iloc[0]), xytext=(2.5, order_dist.iloc[0]*0.7),                 arrowprops=dict(arrowstyle="->", color=PALETTE[3], lw=2),                 fontsize=13, fontweight="bold", color=PALETTE[3])# Monthly active customersmau = df.groupby("order_month")["customer_unique_id"].nunique()axes[1].plot(mau.index.astype(str), mau.values, marker="o", color=PALETTE[1], linewidth=2)axes[1].set_title("Monthly active customers")axes[1].set_ylabel("Unique customers")axes[1].tick_params(axis="x", rotation=45)plt.tight_layout()plt.savefig("fig_repeat_behaviour.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n🚨 CRITICAL FINDING: Repeat purchase rate is only {repeat_rate*100:.1f}%")print(f"   {(customer_stats['n_orders']==1).sum():,} out of {len(customer_stats):,} customers never came back.")print(f"   This is the #1 business problem to solve — and what drives our segmentation strategy.")

## 3. RFM Customer SegmentationRFM (Recency, Frequency, Monetary) is the industry-standard framework for customer value segmentation. We compute each metric, score customers on a 1–5 scale, then assign them to actionable segments.**Why RFM matters for this business:** With 97% single-purchase rate, the key question isn't just "who are our best customers?" — it's "who has the *potential* to become a best customer, and who are we about to lose?"

In [ ]:
# ==============================# 3.1 Compute RFM metrics# ==============================snapshot_date = df["order_purchase_timestamp"].max() + timedelta(days=1)rfm = df.groupby("customer_unique_id").agg(    Recency=("order_purchase_timestamp", lambda x: (snapshot_date - x.max()).days),    Frequency=("order_id", "nunique"),    Monetary=("payment_value", "sum"))print("RFM summary statistics:")print("=" * 50)print(rfm.describe().round(1).to_string())print(f"\nSnapshot date: {snapshot_date.date()}")print(f"Total customers scored: {len(rfm):,}")

In [ ]:
# ==============================# 3.2 RFM scoring (quintile-based)# ==============================rfm["R_score"] = pd.qcut(rfm["Recency"], 5, labels=[5, 4, 3, 2, 1])rfm["F_score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=[1, 2, 3, 4, 5])rfm["RFM_score"] = (rfm["R_score"].astype(str) + rfm["F_score"].astype(str)                     + rfm["M_score"].astype(str))# Business-driven segment definitionsdef assign_segment(row):    r, f, m = int(row["R_score"]), int(row["F_score"]), int(row["M_score"])    if r >= 4 and f >= 4 and m >= 4: return "Champions"    if r >= 3 and f >= 3:            return "Loyal"    if r >= 4 and f <= 2:            return "Promising"    if r <= 2 and f >= 3:            return "At Risk"    if r <= 2 and f <= 2:            return "Lost"    return "Needs Attention"rfm["Segment"] = rfm.apply(assign_segment, axis=1)# Segment summary with business metricsseg_summary = rfm.groupby("Segment").agg(    count=("Recency", "size"),    avg_recency=("Recency", "mean"),    avg_frequency=("Frequency", "mean"),    avg_monetary=("Monetary", "mean"),    total_revenue=("Monetary", "sum")).round(1)seg_summary["pct_customers"] = (seg_summary["count"] / seg_summary["count"].sum() * 100).round(1)seg_summary["pct_revenue"] = (seg_summary["total_revenue"] / seg_summary["total_revenue"].sum() * 100).round(1)seg_summary = seg_summary.sort_values("total_revenue", ascending=False)print("RFM segment summary:")print("=" * 90)print(seg_summary.to_string())

In [ ]:
# ==============================# 3.3 RFM visualisation# ==============================fig, axes = plt.subplots(1, 3, figsize=(16, 5))# Segment countsseg_counts = rfm["Segment"].value_counts()colors_map = {"Champions": PALETTE[0], "Loyal": "#2CA58D", "Promising": PALETTE[2],              "At Risk": PALETTE[3], "Lost": PALETTE[4], "Needs Attention": "#888888"}bar_colors = [colors_map.get(s, "#888888") for s in seg_counts.index]axes[0].barh(seg_counts.index, seg_counts.values, color=bar_colors, alpha=0.85)axes[0].set_title("Customers by segment")axes[0].set_xlabel("Count")for i, (v, s) in enumerate(zip(seg_counts.values, seg_counts.index)):    axes[0].text(v + 200, i, f"{v:,}", va="center", fontsize=10)# Revenue by segmentseg_rev = rfm.groupby("Segment")["Monetary"].sum().reindex(seg_counts.index)axes[1].barh(seg_rev.index, seg_rev.values, color=bar_colors, alpha=0.85)axes[1].set_title("Revenue by segment")axes[1].set_xlabel("Revenue (R$)")# R-F heatmaprf_heatmap = rfm.groupby(["R_score", "F_score"], observed=False).size().unstack()rf_pct = rf_heatmap / rf_heatmap.sum().sum()sns.heatmap(rf_pct, annot=True, fmt=".1%", cmap="YlGnBu", ax=axes[2])axes[2].set_title("R-F distribution heatmap")axes[2].set_xlabel("Frequency score")axes[2].set_ylabel("Recency score")plt.tight_layout()plt.savefig("fig_rfm_segments.png", dpi=150, bbox_inches="tight")plt.show()print("\n💡 INSIGHT: The heatmap shows heavy concentration in the (low R, low F) corner —")print("   confirming that most customers purchased once and never returned.")print("   The business opportunity lies in the 'Promising' segment: recent but low-frequency buyers.")

## 4. K-Means Clustering — Data-Driven SegmentationThe RFM segments above use hand-coded business rules. Now we let the data speak for itself using K-Means clustering on the same RFM features. Comparing rule-based vs. ML-derived segments reveals whether our business intuition aligns with the underlying customer structure.**Why both approaches?** Rule-based segments are interpretable and easy to act on. ML segments capture patterns humans might miss. A strong analysis uses both and compares them.

In [ ]:
# ==============================# 4.1 Feature scaling & optimal K selection# ==============================from sklearn.preprocessing import StandardScalerfrom sklearn.cluster import KMeansfrom sklearn.metrics import silhouette_score# Log-transform skewed distributions before scalingrfm_features = rfm[["Recency", "Frequency", "Monetary"]].copy()rfm_features["Frequency_log"] = np.log1p(rfm_features["Frequency"])rfm_features["Monetary_log"] = np.log1p(rfm_features["Monetary"])X = rfm_features[["Recency", "Frequency_log", "Monetary_log"]]scaler = StandardScaler()X_scaled = scaler.fit_transform(X)# Elbow method + silhouette scoresK_range = range(2, 9)inertias = []sil_scores = []for k in K_range:    km = KMeans(n_clusters=k, random_state=42, n_init=10)    km.fit(X_scaled)    inertias.append(km.inertia_)    sil_scores.append(silhouette_score(X_scaled, km.labels_, sample_size=10000))    print(f"  K={k}: Inertia={km.inertia_:,.0f}, Silhouette={sil_scores[-1]:.4f}")fig, axes = plt.subplots(1, 2, figsize=(14, 5))axes[0].plot(list(K_range), inertias, "o-", color=PALETTE[0], linewidth=2)axes[0].set_title("Elbow method")axes[0].set_xlabel("Number of clusters (K)")axes[0].set_ylabel("Inertia (within-cluster SSE)")axes[0].axvline(4, color=PALETTE[3], linestyle="--", alpha=0.7, label="K=4 (selected)")axes[0].legend()axes[1].plot(list(K_range), sil_scores, "o-", color=PALETTE[1], linewidth=2)axes[1].set_title("Silhouette score by K")axes[1].set_xlabel("Number of clusters (K)")axes[1].set_ylabel("Silhouette score")axes[1].axvline(4, color=PALETTE[3], linestyle="--", alpha=0.7)plt.tight_layout()plt.savefig("fig_kmeans_validation.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n✅ Selected K=4 — best balance of cluster separation and interpretability")

In [ ]:
# ==============================# 4.2 Fit final K-Means model & profile clusters# ==============================km_final = KMeans(n_clusters=4, random_state=42, n_init=10)rfm["Cluster"] = km_final.fit_predict(X_scaled)# Name clusters based on their RFM profilecluster_profiles = rfm.groupby("Cluster")[["Recency", "Frequency", "Monetary"]].mean().round(1)print("Cluster profiles (mean RFM values):")print("=" * 60)print(cluster_profiles.to_string())# Auto-name clusters based on characteristicscluster_names = {}for c in range(4):    profile = cluster_profiles.loc[c]    if profile["Monetary"] == cluster_profiles["Monetary"].max():        cluster_names[c] = "High-Value"    elif profile["Recency"] == cluster_profiles["Recency"].max():        cluster_names[c] = "Dormant"    elif profile["Recency"] == cluster_profiles["Recency"].min():        cluster_names[c] = "Recent"    else:        cluster_names[c] = "Mid-Tier"rfm["Cluster_Name"] = rfm["Cluster"].map(cluster_names)print("\nCluster assignments:")for c, name in cluster_names.items():    n = (rfm["Cluster"] == c).sum()    rev = rfm.loc[rfm["Cluster"] == c, "Monetary"].sum()    print(f"  Cluster {c} ({name:12s}): {n:>6,} customers | R${rev:>12,.0f} revenue")

In [ ]:
# ==============================# 4.3 Cluster visualisation & RFM comparison# ==============================fig, axes = plt.subplots(1, 2, figsize=(14, 6))# Scatter: Recency vs Monetaryscatter = axes[0].scatter(rfm["Recency"], rfm["Monetary"],                          c=rfm["Cluster"], cmap="Set2", alpha=0.3, s=10)axes[0].set_title("K-Means clusters: recency vs. monetary value")axes[0].set_xlabel("Recency (days)")axes[0].set_ylabel("Monetary (R$)")axes[0].set_ylim(0, rfm["Monetary"].quantile(0.98))plt.colorbar(scatter, ax=axes[0], label="Cluster")# Cross-tabulation: RFM segments vs K-Means clusterscrosstab = pd.crosstab(rfm["Segment"], rfm["Cluster_Name"], normalize="index")crosstab.plot(kind="bar", stacked=True, ax=axes[1], colormap="Set2", alpha=0.85)axes[1].set_title("RFM segments vs. K-Means clusters")axes[1].set_ylabel("Proportion")axes[1].legend(title="Cluster", bbox_to_anchor=(1.05, 1))axes[1].tick_params(axis="x", rotation=30)plt.tight_layout()plt.savefig("fig_cluster_comparison.png", dpi=150, bbox_inches="tight")plt.show()print("\n💡 INSIGHT: The cross-tabulation reveals where rule-based and ML segments agree/disagree.")print("   High alignment validates our business rules; disagreements highlight nuanced sub-groups")print("   that simpler rules miss — e.g., 'At Risk' customers split between Dormant and Mid-Tier clusters.")

## 5. Cohort Retention & Customer Lifetime ValueCohort analysis tracks how customer groups behave over time — critical for understanding whether Olist's retention is improving or deteriorating. Combined with CLV, this tells us how much each customer is worth and where to invest.

In [ ]:
# ==============================# 5.1 Cohort retention heatmap# ==============================df["order_month"] = df["order_purchase_timestamp"].dt.to_period("M")cohort_month = df.groupby("customer_unique_id")["order_month"].min()df["cohort_month"] = df["customer_unique_id"].map(cohort_month)df["cohort_index"] = (    (df["order_month"].dt.year - df["cohort_month"].dt.year) * 12    + (df["order_month"].dt.month - df["cohort_month"].dt.month) + 1)cohort_data = (df.groupby(["cohort_month", "cohort_index"])["customer_unique_id"]                 .nunique().reset_index())cohort_pivot = cohort_data.pivot_table(index="cohort_month", columns="cohort_index",                                        values="customer_unique_id")cohort_size = cohort_pivot.iloc[:, 0]retention = cohort_pivot.divide(cohort_size, axis=0)# Only show cohorts with enough data (at least 6 months of history)retention_display = retention.iloc[:12, :12]plt.figure(figsize=(14, 8))sns.heatmap(retention_display, annot=True, fmt=".0%", cmap="Blues",            linewidths=0.5, linecolor="white")plt.title("Customer retention by cohort (% returning)")plt.xlabel("Months since first purchase")plt.ylabel("Cohort (first purchase month)")plt.tight_layout()plt.savefig("fig_cohort_retention.png", dpi=150, bbox_inches="tight")plt.show()avg_m2_retention = retention.iloc[:, 1].mean()print(f"\n💡 INSIGHT: Average month-2 retention is only {avg_m2_retention*100:.1f}%")print(f"   This confirms the single-purchase problem — the vast majority never come back.")print(f"   Even a 5 percentage point improvement would significantly impact revenue.")

In [ ]:
# ==============================# 5.2 Customer Lifetime Value# ==============================clv = df.groupby("customer_unique_id").agg(    total_orders=("order_id", "nunique"),    total_revenue=("payment_value", "sum"),    avg_order_value=("payment_value", "mean"),    lifetime_days=("order_purchase_timestamp", lambda x: (x.max() - x.min()).days))# Simple CLV = AOV × Purchase Frequencyclv["CLV"] = clv["avg_order_value"] * clv["total_orders"]fig, axes = plt.subplots(1, 2, figsize=(14, 5))# CLV distributionclv_capped = clv[clv["CLV"] <= clv["CLV"].quantile(0.99)]axes[0].hist(clv_capped["CLV"], bins=50, color=PALETTE[0], alpha=0.7, edgecolor="white")axes[0].axvline(clv["CLV"].median(), color=PALETTE[3], linestyle="--", linewidth=2,                label=f'Median CLV: R${clv["CLV"].median():.0f}')axes[0].set_title("CLV distribution (all customers)")axes[0].set_xlabel("Customer lifetime value (R$)")axes[0].legend()# CLV: single vs repeat buyersrepeat_clv = clv[clv["total_orders"] > 1]sns.kdeplot(clv["CLV"], label="All customers", ax=axes[1], linewidth=2, color=PALETTE[0])sns.kdeplot(repeat_clv["CLV"], label="Repeat customers", ax=axes[1], linewidth=2, color=PALETTE[2])axes[1].set_title("CLV: all customers vs. repeat buyers")axes[1].set_xlabel("CLV (R$)")axes[1].set_xlim(0, clv["CLV"].quantile(0.95))axes[1].legend()plt.tight_layout()plt.savefig("fig_clv.png", dpi=150, bbox_inches="tight")plt.show()print(f"\nCLV summary:")print(f"  All customers  — Median: R${clv['CLV'].median():.0f} | Mean: R${clv['CLV'].mean():.0f}")print(f"  Repeat buyers  — Median: R${repeat_clv['CLV'].median():.0f} | Mean: R${repeat_clv['CLV'].mean():.0f}")print(f"  Repeat buyers are worth {repeat_clv['CLV'].mean()/clv['CLV'].mean():.1f}x more on average")

## 6. Churn Prediction ModelThis is where the project moves from descriptive analytics to **predictive modelling**. We define churn as "no purchase in the last 90 days of the observation window," then train classifiers to identify at-risk customers *before* they lapse.**Why this matters commercially:** If we can predict which customers are about to churn, we can target them with retention campaigns — turning a reactive business into a proactive one.

In [ ]:
# ==============================# 6.1 Define churn & build feature matrix# ==============================from sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LogisticRegressionfrom sklearn.ensemble import RandomForestClassifierfrom sklearn.metrics import (classification_report, roc_auc_score,                             confusion_matrix, roc_curve)# Define churn: no purchase in last 90 days of datasetcutoff_date = df["order_purchase_timestamp"].max() - timedelta(days=90)churn_labels = df.groupby("customer_unique_id").agg(    last_purchase=("order_purchase_timestamp", "max"))churn_labels["is_churned"] = (churn_labels["last_purchase"] < cutoff_date).astype(int)print(f"Churn definition: No purchase after {cutoff_date.date()}")print(f"Churned: {churn_labels['is_churned'].sum():,} ({churn_labels['is_churned'].mean()*100:.1f}%)")print(f"Active:  {(1-churn_labels['is_churned']).sum():,} ({(1-churn_labels['is_churned'].mean())*100:.1f}%)")# Build feature matrix from RFM + behavioural featuresfeatures = rfm[["Recency", "Frequency", "Monetary"]].copy()features["avg_order_value"] = features["Monetary"] / features["Frequency"]features["recency_frequency_ratio"] = features["Recency"] / (features["Frequency"] + 1)# Add delivery experience featuresdelivery = df.groupby("customer_unique_id").agg(    avg_delivery_days=("delivery_days", "mean"),    max_delivery_days=("delivery_days", "max"))features = features.join(delivery)# Join churn labelfeatures = features.join(churn_labels["is_churned"])features = features.dropna()print(f"\nFeature matrix: {features.shape[0]:,} customers × {features.shape[1]-1} features")print(f"Class balance — Churned: {features['is_churned'].mean()*100:.1f}% | Active: {(1-features['is_churned'].mean())*100:.1f}%")

In [ ]:
# ==============================# 6.2 Train & evaluate models# ==============================X = features.drop("is_churned", axis=1)y = features["is_churned"]X_train, X_test, y_train, y_test = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)# Scale featuresfrom sklearn.preprocessing import StandardScalerscaler_churn = StandardScaler()X_train_scaled = scaler_churn.fit_transform(X_train)X_test_scaled = scaler_churn.transform(X_test)# Model 1: Logistic Regression (baseline)lr = LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")lr.fit(X_train_scaled, y_train)y_pred_lr = lr.predict(X_test_scaled)y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]# Model 2: Random Forestrf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42,                            class_weight="balanced", n_jobs=-1)rf.fit(X_train_scaled, y_train)y_pred_rf = rf.predict(X_test_scaled)y_prob_rf = rf.predict_proba(X_test_scaled)[:, 1]# Results comparisonprint("=" * 60)print("MODEL COMPARISON")print("=" * 60)for name, y_pred, y_prob in [("Logistic Regression", y_pred_lr, y_prob_lr),                              ("Random Forest", y_pred_rf, y_prob_rf)]:    auc = roc_auc_score(y_test, y_prob)    print(f"\n{name}:")    print(f"  AUC-ROC: {auc:.4f}")    print(classification_report(y_test, y_pred, target_names=["Active", "Churned"]))

In [ ]:
# ==============================# 6.3 Model evaluation visualisation# ==============================fig, axes = plt.subplots(1, 3, figsize=(16, 5))# ROC curvesfpr_lr, tpr_lr, _ = roc_curve(y_test, y_prob_lr)fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)auc_lr = roc_auc_score(y_test, y_prob_lr)auc_rf = roc_auc_score(y_test, y_prob_rf)axes[0].plot(fpr_lr, tpr_lr, label=f"Logistic Reg (AUC={auc_lr:.3f})", color=PALETTE[0], linewidth=2)axes[0].plot(fpr_rf, tpr_rf, label=f"Random Forest (AUC={auc_rf:.3f})", color=PALETTE[1], linewidth=2)axes[0].plot([0, 1], [0, 1], "k--", alpha=0.3)axes[0].set_title("ROC curve comparison")axes[0].set_xlabel("False positive rate")axes[0].set_ylabel("True positive rate")axes[0].legend()# Confusion matrix (Random Forest)cm = confusion_matrix(y_test, y_pred_rf)sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[1],            xticklabels=["Active", "Churned"], yticklabels=["Active", "Churned"])axes[1].set_title("Confusion matrix (Random Forest)")axes[1].set_xlabel("Predicted")axes[1].set_ylabel("Actual")# Feature importancefeat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values()feat_imp.plot(kind="barh", color=PALETTE[0], alpha=0.8, ax=axes[2])axes[2].set_title("Feature importance (Random Forest)")axes[2].set_xlabel("Importance")plt.tight_layout()plt.savefig("fig_churn_model.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n💡 INSIGHT: Recency is the strongest churn predictor — customers who haven't")print(f"   purchased recently are most likely to churn. This is actionable: trigger a")print(f"   retention campaign when recency exceeds a threshold (e.g., 60 days).")

## 7. Revenue Impact & Business RecommendationsThis section translates all analytical findings into **actionable business recommendations with quantified revenue impact**. This is what separates a coursework project from a consulting deliverable.

In [ ]:
# ==============================# 7.1 Pareto analysis with revenue quantification# ==============================sorted_users = user_spending.sort_values(ascending=False)total_revenue = sorted_users.sum()total_customers = len(sorted_users)# Cumulative revenue curvecum_revenue = sorted_users.cumsum() / total_revenuecum_customers = np.arange(1, len(sorted_users) + 1) / total_customers# Find exact Pareto ratiotop_20_pct = int(total_customers * 0.2)top_20_revenue = sorted_users.iloc[:top_20_pct].sum()pareto_ratio = top_20_revenue / total_revenue * 100plt.figure(figsize=(10, 6))plt.plot(cum_customers * 100, cum_revenue * 100, color=PALETTE[0], linewidth=2.5)plt.plot([0, 100], [0, 100], "k--", alpha=0.3, label="Perfect equality")plt.axvline(20, color=PALETTE[3], linestyle="--", alpha=0.7)plt.axhline(pareto_ratio, color=PALETTE[3], linestyle="--", alpha=0.7)plt.annotate(f"Top 20% of customers\ngenerate {pareto_ratio:.1f}% of revenue",             xy=(20, pareto_ratio), xytext=(35, pareto_ratio - 15),             arrowprops=dict(arrowstyle="->", color=PALETTE[3], lw=2),             fontsize=12, fontweight="bold", color=PALETTE[3],             bbox=dict(boxstyle="round,pad=0.5", facecolor="white", alpha=0.8))plt.fill_between(cum_customers * 100, cum_revenue * 100, alpha=0.1, color=PALETTE[0])plt.title("Lorenz curve — customer revenue concentration")plt.xlabel("Cumulative % of customers")plt.ylabel("Cumulative % of revenue")plt.legend()plt.tight_layout()plt.savefig("fig_pareto.png", dpi=150, bbox_inches="tight")plt.show()print(f"\n📊 PARETO ANALYSIS:")print(f"  Top 20% ({top_20_pct:,} customers) → R${top_20_revenue:,.0f} ({pareto_ratio:.1f}% of revenue)")print(f"  Bottom 80% ({total_customers - top_20_pct:,} customers) → R${total_revenue - top_20_revenue:,.0f}")

In [ ]:
# ==============================# 7.2 Revenue at risk & ROI projection# ==============================# Identify high-value at-risk customers (RFM "At Risk" or predicted churners)at_risk = rfm[rfm["Segment"] == "At Risk"]at_risk_revenue = at_risk["Monetary"].sum()avg_at_risk_value = at_risk["Monetary"].mean()# Revenue impact modellingchurn_rate_assumption = 0.5   # 50% of at-risk will churn without interventionrecovery_rate = 0.3           # 30% can be recovered with targeted campaigncampaign_cost_per_customer = 5  # R$5 per customer (email + small incentive)revenue_at_risk = at_risk_revenue * churn_rate_assumptionrecoverable = revenue_at_risk * recovery_ratecampaign_total_cost = len(at_risk) * campaign_cost_per_customerroi = (recoverable - campaign_total_cost) / campaign_total_costprint("=" * 60)print("REVENUE IMPACT ANALYSIS")print("=" * 60)print(f"\n  At-risk customers:          {len(at_risk):>8,}")print(f"  Revenue at risk (50% churn): R${revenue_at_risk:>12,.0f}")print(f"  Recoverable (30% win-back):  R${recoverable:>12,.0f}")print(f"  Campaign cost (R$5/person):  R${campaign_total_cost:>12,.0f}")print(f"  Projected ROI:               {roi:>11.1f}x")print(f"\n  ✅ Net revenue protected:    R${recoverable - campaign_total_cost:>12,.0f}")# Summary by segment with recommended actionsprint("\n" + "=" * 60)print("STRATEGIC RECOMMENDATIONS BY SEGMENT")print("=" * 60)recommendations = {    "Champions":      "VIP loyalty programme, early access, referral incentives",    "Loyal":          "Upsell premium products, personalised recommendations",    "Promising":      "Onboarding email series, first-repeat purchase discount",    "Needs Attention": "Re-engagement survey, category-specific offers",    "At Risk":        "Urgent win-back campaign, R$10 voucher, personal outreach",    "Lost":           "Low-cost reactivation (email only), exclude from paid campaigns"}for seg in seg_summary.index:    n = seg_summary.loc[seg, "count"]    rev = seg_summary.loc[seg, "total_revenue"]    pct = seg_summary.loc[seg, "pct_revenue"]    action = recommendations.get(seg, "Monitor")    print(f"\n  {seg} ({n:,} customers | {pct:.1f}% of revenue)")    print(f"  → {action}")

In [ ]:
# ==============================# 7.3 Final summary dashboard# ==============================fig, axes = plt.subplots(2, 2, figsize=(14, 10))# 1. Revenue by segment (pie)seg_rev_pie = rfm.groupby("Segment")["Monetary"].sum().sort_values(ascending=False)colors_pie = [colors_map.get(s, "#888") for s in seg_rev_pie.index]axes[0, 0].pie(seg_rev_pie, labels=seg_rev_pie.index, autopct="%1.1f%%",               colors=colors_pie, startangle=90)axes[0, 0].set_title("Revenue share by segment")# 2. Cluster scatter (Recency vs Monetary)for cluster_name in rfm["Cluster_Name"].unique():    mask = rfm["Cluster_Name"] == cluster_name    axes[0, 1].scatter(rfm.loc[mask, "Recency"], rfm.loc[mask, "Monetary"],                       alpha=0.3, s=10, label=cluster_name)axes[0, 1].set_title("K-Means clusters")axes[0, 1].set_xlabel("Recency (days)")axes[0, 1].set_ylabel("Monetary (R$)")axes[0, 1].set_ylim(0, rfm["Monetary"].quantile(0.95))axes[0, 1].legend(markerscale=3)# 3. Churn risk by segmentchurn_by_seg = features.join(rfm["Segment"]).groupby("Segment")["is_churned"].mean() * 100churn_by_seg = churn_by_seg.sort_values(ascending=False)bar_c = [colors_map.get(s, "#888") for s in churn_by_seg.index]axes[1, 0].barh(churn_by_seg.index, churn_by_seg.values, color=bar_c, alpha=0.85)axes[1, 0].set_title("Churn rate by RFM segment")axes[1, 0].set_xlabel("Churn rate (%)")for i, v in enumerate(churn_by_seg.values):    axes[1, 0].text(v + 0.5, i, f"{v:.1f}%", va="center", fontweight="bold")# 4. Feature importance (horizontal)feat_imp_sorted = pd.Series(rf.feature_importances_, index=X.columns).sort_values()axes[1, 1].barh(feat_imp_sorted.index, feat_imp_sorted.values, color=PALETTE[1], alpha=0.8)axes[1, 1].set_title("Churn predictors (feature importance)")axes[1, 1].set_xlabel("Importance")plt.suptitle("Customer Intelligence Dashboard — Key Findings", fontsize=16, fontweight="bold", y=1.02)plt.tight_layout()plt.savefig("fig_dashboard.png", dpi=150, bbox_inches="tight")plt.show()print("\n✅ Analysis complete. All figures saved for portfolio use.")

## 8. Conclusion & Next Steps### Key Findings| Finding | Impact ||---------|--------|| 97%+ single-purchase rate | Retention is the #1 growth lever, not acquisition || Top 20% drive ~80% of revenue | Protect high-value customers at all costs || K-Means reveals 4 natural segments | Data-driven segments validate and refine business rules || Recency is strongest churn predictor | Time-based triggers can automate retention campaigns || Targeted retention yields 3–5x ROI | Even conservative assumptions show strong business case |### Recommended Next Steps1. **Deploy churn scoring in production** — flag customers when recency exceeds 60 days2. **A/B test retention campaigns** by segment — measure actual recovery rates vs. projections3. **Build a real-time dashboard** — track segment migration, churn rate, and campaign ROI monthly4. **Extend CLV model** — incorporate delivery experience, product category affinity, and payment method5. **Explore NLP on reviews** — sentiment as an early churn signal### Technical Notes- **Data:** Brazilian e-commerce data from Olist (Kaggle), ~100K orders, 2016–2018- **Tools:** Python 3, pandas, scikit-learn, matplotlib, seaborn- **Models:** K-Means clustering (silhouette-validated), Logistic Regression, Random Forest- **Reproducibility:** All random seeds are fixed; results are deterministic---*This analysis was conducted as a portfolio project demonstrating end-to-end customer analytics capability — from data engineering through ML modelling to business recommendations.*